# Stage 2 - Cleaning the DVF sales

**Input:** `data/raw/dvf/dvf_74_<year>.csv.gz` (from `01_get_data.py`)
**Output:** `data/processed/sales_clean.csv` (one row per sale) and `data/processed/cleaning_log.csv` (how many sales each rule removed)

DVF is not one row per sale. One sale (`id_mutation`) is spread over several rows: one per building unit (home, garage, cellar, shop) and one per type of land. The goal of this notebook is to end with **one row = one home sold alone, with a reliable price per square metre**.

*Tool declaration: this notebook was drafted with Claude (Anthropic), asked to write the DVF cleaning steps described below.*

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from helpers import DEPARTMENT, PROCESSED, RAW, YEARS

# Cleaning thresholds: every choice is here, in one place
KEPT_SALE_TYPES = ["Vente", "Vente en l'état futur d'achèvement"]  # normal sales + new builds sold off-plan
HOME_TYPES = ["Maison", "Appartement"]
SHOP_TYPE = "Local industriel. commercial ou assimilé"
MIN_SURFACE, MAX_SURFACE = 9, 400            # m2 (9 m2 is the legal minimum for a dwelling in France)
MIN_PRICE_M2, MAX_PRICE_M2 = 1_000, 20_000   # euros per m2

## 1. Load the raw files

In [ ]:
files = [RAW / "dvf" / f"dvf_{DEPARTMENT}_{year}.csv.gz" for year in YEARS]
raw = pd.concat(
    [pd.read_csv(f, dtype={"code_commune": str}, low_memory=False) for f in files],
    ignore_index=True,
)
print(f"{len(raw):,} rows, {raw['id_mutation'].nunique():,} sales, {raw.shape[1]} columns")
raw["type_local"].value_counts(dropna=False)

A small helper to keep track of what each rule removes. The resulting table goes into the paper.

In [ ]:
log = []

def record(step, df):
    """Store the number of rows and of distinct sales left after a cleaning step."""
    log.append({"step": step, "rows": len(df), "sales": df["id_mutation"].nunique()})
    print(f"{step}: {len(df):,} rows, {df['id_mutation'].nunique():,} sales")

record("Raw data", raw)

## 2. Exact duplicates

Some rows are repeated identically (same sale, same unit, same land). They carry no information.

In [ ]:
sales = raw.drop_duplicates()
record("Drop exact duplicate rows", sales)

## 3. Keep real sales only

Exchanges, auctions, expropriations and building-land sales do not have a normal market price. I keep ordinary sales and new builds sold off-plan (VEFA), and flag the latter with `new_build` so the models can control for it.

In [ ]:
sales = sales[sales["nature_mutation"].isin(KEPT_SALE_TYPES)]
record("Keep ordinary sales and off-plan sales", sales)

## 4. One row per building unit

When a parcel has several types of land (garden, yard...), DVF repeats each building unit once per land type. I first add up the land of each sale, then drop the land columns and remove the repeats, which leaves one row per building unit.

In [ ]:
land_columns = ["code_nature_culture", "nature_culture", "code_nature_culture_speciale",
                "nature_culture_speciale", "surface_terrain"]

# Land surface of each sale: each (parcel, land type) counted once
land = (
    sales.drop_duplicates(["id_mutation", "id_parcelle", "code_nature_culture",
                           "code_nature_culture_speciale", "surface_terrain"])
    .groupby("id_mutation")["surface_terrain"].sum()
    .rename("land_surface")
)

units = sales[sales["type_local"].notna()].drop(columns=land_columns).drop_duplicates()
record("One row per building unit (land-only rows removed)", units)

## 5. Keep sales of exactly one home

The price of a sale is given for the whole sale, not per unit. If two flats are sold together, or a flat with a shop, the price cannot be split, so the price per square metre is meaningless. I keep sales with **exactly one house or flat and no commercial unit**. Outbuildings (garage, cellar) are allowed: I count them, because a flat sold with a garage is worth more.

In [ ]:
counts = pd.crosstab(units["id_mutation"], units["type_local"])
n_homes = counts[HOME_TYPES].sum(axis=1)
single_home = counts.index[(n_homes == 1) & (counts[SHOP_TYPE] == 0)]

print("Number of homes per sale:")
print(n_homes.value_counts().sort_index().head(6))

homes = units[units["id_mutation"].isin(single_home) & units["type_local"].isin(HOME_TYPES)].copy()
homes["n_outbuildings"] = homes["id_mutation"].map(counts["Dépendance"])
homes["land_surface"] = homes["id_mutation"].map(land).fillna(0)
assert homes["id_mutation"].is_unique  # from here on: one row = one sale
record("Keep sales of exactly one home, no shop", homes)

## 6. Missing price, missing location

In [ ]:
homes = homes[homes["valeur_fonciere"].notna()]
record("Drop missing price", homes)

homes = homes[homes["latitude"].notna() & homes["longitude"].notna()]
record("Drop missing coordinates", homes)

## 7. Implausible surfaces and prices

Some sales have a symbolic price (1 euro, family transfers) or a wrong surface. I drop surfaces outside 9-400 m2 and prices outside 1,000-20,000 euros per m2. The upper bound removes a few hundred genuine luxury sales, mostly in Megève and Chamonix; this is a choice, and the bound is a constant in the first cell so it can be changed to check robustness.

In [ ]:
homes["price_m2"] = homes["valeur_fonciere"] / homes["surface_reelle_bati"]
print(homes["price_m2"].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).round(0))

homes = homes[homes["surface_reelle_bati"].between(MIN_SURFACE, MAX_SURFACE)]
record(f"Keep surface between {MIN_SURFACE} and {MAX_SURFACE} m2", homes)

too_expensive = homes[homes["price_m2"] > MAX_PRICE_M2]
print("\nMunicipalities most affected by the upper price bound:")
print(too_expensive["nom_commune"].value_counts().head(5))

homes = homes[homes["price_m2"].between(MIN_PRICE_M2, MAX_PRICE_M2)]
record(f"Keep price between {MIN_PRICE_M2:,} and {MAX_PRICE_M2:,} euros/m2", homes)

## 8. Final table

In [ ]:
homes["date"] = pd.to_datetime(homes["date_mutation"])

clean = pd.DataFrame({
    "id_mutation": homes["id_mutation"],
    "date": homes["date"].dt.strftime("%Y-%m-%d"),
    "year": homes["date"].dt.year,
    "quarter": homes["date"].dt.to_period("Q").astype(str),   # e.g. 2023Q2
    "price": homes["valeur_fonciere"],
    "surface": homes["surface_reelle_bati"],
    "price_m2": homes["price_m2"].round(1),
    "is_house": (homes["type_local"] == "Maison").astype(int),
    "rooms": homes["nombre_pieces_principales"],
    "land_surface": homes["land_surface"],
    "n_outbuildings": homes["n_outbuildings"],
    "new_build": (homes["nature_mutation"] != "Vente").astype(int),
    "code_commune": homes["code_commune"],
    "commune": homes["nom_commune"],
    "latitude": homes["latitude"],
    "longitude": homes["longitude"],
}).sort_values(["date", "id_mutation"]).reset_index(drop=True)

print(clean.isna().sum()[clean.isna().sum() > 0])  # remaining missing values, if any
clean.head()

In [ ]:
PROCESSED.mkdir(parents=True, exist_ok=True)
clean.to_csv(PROCESSED / "sales_clean.csv", index=False)

cleaning_log = pd.DataFrame(log)
cleaning_log["sales_removed"] = -cleaning_log["sales"].diff().fillna(0).astype(int)
cleaning_log.to_csv(PROCESSED / "cleaning_log.csv", index=False)
cleaning_log

## 9. Quick checks on the clean data

In [ ]:
summary = clean.groupby("year").agg(
    sales=("price_m2", "size"),
    median_price_m2=("price_m2", "median"),
    share_houses=("is_house", "mean"),
    share_new_build=("new_build", "mean"),
).round(2)
print(f"{len(clean):,} sales in {clean['code_commune'].nunique()} municipalities")
summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

clean["price_m2"].hist(bins=80, ax=axes[0])
axes[0].set_xlabel("Price per m2 (euros)")
axes[0].set_ylabel("Number of sales")
axes[0].set_title("Distribution of prices after cleaning")

clean.groupby("quarter")["price_m2"].median().plot(ax=axes[1], marker="o")
axes[1].set_xlabel("Quarter of sale")
axes[1].set_ylabel("Median price per m2 (euros)")
axes[1].set_title("Median price by quarter")

plt.tight_layout()
plt.show()